# 00 · Start here: Jupyter, tensors, and your GPU

**Level:** first contact with PyTorch · **Teaching time:** 30–45 minutes.

By the end you will run notebook cells, verify your GPU, read image tensor shapes,
calculate class probabilities, and watch automatic differentiation work.
You need basic Python variables, lists, functions, and loops; no deep learning experience is assumed.

This course contains **13 independent notebooks**. Work through 00–12 in order when learning;
later you can run any single lesson from a clean kernel for a demonstration.
Keep the whole folder together: `course_utils.py` contains shared data and plotting helpers.

**Before class:** use the PowerShell setup in the root README, select the
**Image Classification (PyTorch)** kernel, and run notebooks 01 and 06 once to cache their downloads.


## Step 1 — Learn the notebook controls

Click a cell and press **Shift+Enter** to run it and move down. Markdown cells contain explanations;
code cells execute Python in one shared kernel. An `[*]` prompt means a cell is still running.
Use **Kernel → Restart Kernel and Run All Cells** to verify that a lesson works from top to bottom.
Running cells out of order can leave old values in memory, so restart after changing the configuration.

**Demonstrate:** change the greeting below, run it, then explain that Python keeps its value until restart.


In [ ]:
greeting = "Welcome to practical image classification!"
print(greeting)


## Step 2 — Check Python and the selected kernel

The interpreter path should point to this course's `.venv` when using the supplied launcher.
Installing a package into one Python does not make it available to every Jupyter kernel.


In [ ]:
import sys
import platform
print("Python:", sys.version)
print("Interpreter:", sys.executable)
print("Platform:", platform.platform())


In [ ]:
from pathlib import Path
import sys

# Works when Jupyter starts in the project root or in notebooks/.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "course_utils.py").exists()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from the complete course folder; course_utils.py is required.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib.pyplot as plt
import torch
import course_utils as cu

DEVICE = cu.configure(seed=42)
SMOKE = cu.SMOKE  # COURSE_SMOKE=1: validation only; too small to judge model quality.
QUICK = True     # Set False for a longer learning experiment, then Restart Kernel and Run All.


## Step 3 — Prove the GPU can perform an operation

CUDA is the interface PyTorch uses to run on an NVIDIA GPU. A CPU tensor must be moved to the
GPU before a GPU model can process it. The model and its input must be on the same device.
`nvidia-smi` reports the driver's CUDA compatibility; `torch.version.cuda` reports the runtime
used to build PyTorch. They do not have to show the same version.

The code below also works on a CPU, but the device label tells you which hardware actually ran it.


In [ ]:
print("CUDA available:", torch.cuda.is_available())
print("PyTorch CUDA runtime:", torch.version.cuda)
a = torch.randn(128, 128, device=DEVICE)
b = a @ a.T
if DEVICE.type == "cuda":
    torch.cuda.synchronize()
assert b.shape == (128, 128) and torch.isfinite(b).all()
print("Matrix multiplication succeeded on", b.device)


## Step 4 — Read the course map

| Lesson | Practical result |
|---|---|
| 00 | Working environment and tensor basics |
| 01 | Visualized images, batches, and leakage-free splits |
| 02 | A trained linear image classifier |
| 03 | A multilayer neural network and a full training loop |
| 04 | A CNN and visualized feature maps |
| 05 | Augmentation comparison, confusion matrix, and error analysis |
| 06 | Pretrained ResNet18: feature extraction and fine-tuning |
| 07 | Your own image-folder dataset, with a runnable shapes example |
| 08 | Validation-only tuning, mixed precision, and resumable training |
| 09 | Saliency, Grad-CAM, and occlusion explanations |
| 10 | Patch embeddings and a trained small vision transformer |
| 11 | Calibration and controlled distribution-shift experiments |
| 12 | Saved model bundle, image prediction, export, and capstone |

These notebooks introduce advanced workflows; becoming expert also requires repeated experiments
on representative real data. Short demo training is for understanding behavior, not claiming benchmark results.


## Step 5 — A tensor is an array with a shape and a data type

An image can be represented as numbers. PyTorch vision models typically receive
`[batch, channels, height, width]`, abbreviated **NCHW**. RGB has three channels; grayscale has one.
Floating-point inputs carry pixel values; integer class labels identify categories.


In [ ]:
single_image = torch.rand(1, 28, 28)  # one grayscale image, values from 0 to 1
batch = torch.stack([single_image, single_image])
labels = torch.tensor([3, 7], dtype=torch.long)
print("Single image CHW:", single_image.shape)
print("Batch NCHW:", batch.shape, batch.dtype)
print("Labels N:", labels.shape, labels.dtype)
print("Flattened batch:", batch.flatten(start_dim=1).shape)
plt.imshow(single_image[0], cmap="gray", vmin=0, vmax=1)
plt.title("Random pixels: useful for shapes, not classification")
plt.axis("off")
plt.show()


## Step 6 — From scores to class probabilities

A classifier returns one score, or **logit**, per class. `softmax` turns those scores into positive
numbers that add to one. `argmax` chooses the class with the greatest score.
These are model probabilities, not a guarantee that a prediction is correct.


In [ ]:
logits = torch.tensor([[2.0, 0.5, -1.0]])
probabilities = logits.softmax(dim=1)
names = ["shirt", "shoe", "bag"]
print("Logits:", logits)
print("Probabilities:", probabilities)
print("Sum:", probabilities.sum(1))
print("Predicted class:", names[logits.argmax(1).item()])
plt.bar(names, probabilities[0].numpy())
plt.ylim(0, 1)
plt.ylabel("Model probability")
plt.show()


## Step 7 — Measure a mistake with cross-entropy

Cross-entropy penalizes giving low probability to the correct class. For one example it is
`-log(probability of the true class)`. PyTorch's `CrossEntropyLoss` accepts **raw logits** and
integer labels; it performs the needed normalization internally. Do not add softmax before it.


In [ ]:
loss_fn = torch.nn.CrossEntropyLoss()
target = torch.tensor([0])
loss = loss_fn(logits, target)
manual_loss = -torch.log(probabilities[0, target.item()])
assert torch.allclose(loss, manual_loss)
print("Loss when shirt is correct:", loss.item())
print("Loss when bag is correct:", loss_fn(logits, torch.tensor([2])).item())


## Step 8 — See the gradient that makes learning possible

Training adjusts weights to lower a loss. A gradient tells us how a small weight change affects that
loss. `backward()` computes gradients; the optimizer later applies an update.
For this scalar example the derivative of `(w - 3)^2` is `2 * (w - 3)`.


In [ ]:
w = torch.tensor(1.0, requires_grad=True)
toy_loss = (w - 3) ** 2
toy_loss.backward()
print("Weight:", w.item(), "loss:", toy_loss.item(), "gradient:", w.grad.item())
assert w.grad.item() == -4.0
with torch.no_grad():
    w -= 0.1 * w.grad
print("Updated weight:", w.item(), "new loss:", ((w - 3) ** 2).item())


## Step 9 — Know the three modes used throughout the course

| Situation | Model mode | Gradient recording |
|---|---|---|
| Training | `model.train()` | Enabled |
| Validation and ordinary prediction | `model.eval()` | `torch.no_grad()` or `torch.inference_mode()` |
| Gradient-based explanations | `model.eval()` | Enabled |

`eval()` changes layers such as dropout and BatchNorm; it does **not** disable gradients.
`no_grad()` prevents recording gradients; it does **not** change training mode.


## Step 10 — Choose demonstration settings

Each training notebook begins with small settings suitable for a live lesson. `QUICK=True` uses
subsets and a few epochs (full passes through training examples). Set `QUICK=False` for a longer run.
The separate `COURSE_SMOKE=1` environment variable is used by the validation script to check tiny
runs. Smoke results are not meaningful model-quality measurements.

Data are cached under `data/`, generated models under `artifacts/`, and downloaded pretrained
weights in PyTorch's user cache. Downloads need internet the first time.
No notebook needs an API key or a paid service.


## Exercises and live questions

1. Make a batch of four RGB images of size 32×32. Predict its shape before printing it.
2. Change the correct class in Step 7. Why does the loss change even though the prediction does not?
3. Set the learning rate in Step 8 to `1.1`. Does this update still improve the loss?

<details><summary>Suggested answers</summary>

1. `torch.rand(4, 3, 32, 32)` has shape `[4, 3, 32, 32]`.
2. Loss compares logits with the supplied target; the prediction uses only the logits.
3. The updated weight is `5.4`, so the loss becomes `5.76`, larger than `4.0`. A larger step is not always better.

</details>

**Ready to move on:** you can explain a batch shape, a logit, a class label, and why we need gradients.

**Troubleshooting:** restart the kernel after installation; reduce batch size on CUDA out-of-memory;
use `num_workers=0` in Windows notebooks; check `sys.executable` when an import fails.
See the README for installation and the demonstration guide for a teaching schedule.

**Primary references:** [PyTorch basics](https://docs.pytorch.org/tutorials/beginner/basics/intro.html),
[local installation](https://pytorch.org/get-started/locally/),
[autograd](https://docs.pytorch.org/tutorials/beginner/basics/autogradqs_tutorial.html).
